In [5]:
from manim import *
import numpy as np

class FourierConcept(Scene):
    def construct(self):
        # ====================
        # 1. LAYOUT & TITLE
        # ====================
        title = Text("The Complete Guide to Fourier Series", font_size=32, weight=BOLD).to_edge(UP, buff=0.2)
        self.play(Write(title))

        left_center = LEFT * 3.5 + UP * 1.0
        right_center = RIGHT * 3.2 
        
        # ====================
        # 2. AXES SETUP 
        # ====================
        time_axes = Axes(
            x_range=[0, 3.5, 1], y_range=[-3, 3, 1],
            x_length=6, y_length=2.5,
            axis_config={"include_tip": False, "font_size": 16}
        ).move_to(right_center + UP * 1.0).add_coordinates()
        time_labels = time_axes.get_axis_labels(x_label="t", y_label="f(t)")

        freq_axes = Axes(
            x_range=[0, 12, 2], y_range=[0, 2.5, 0.5],
            x_length=6, y_length=2.2,
            axis_config={"include_tip": False, "font_size": 16}
        ).move_to(right_center + DOWN * 2.2).add_coordinates()
        freq_labels = freq_axes.get_axis_labels(x_label="Hz", y_label="Amp")

        self.play(Create(time_axes), Write(time_labels))

        def create_desc(text_string):
            return Text(text_string, font_size=16, line_spacing=1.2).move_to(left_center)

        # ====================
        # 3. PHASE 1: SINE WAVE ANATOMY (INDIVIDUAL & MIXED)
        # ====================
        amp_tracker = ValueTracker(1.0)
        freq_tracker = ValueTracker(1.0)
        phase_tracker = ValueTracker(0.0)

        desc1 = create_desc(
            "1. Wave Anatomy\n\n"
            "Every sine wave is defined by:\n"
            "- Amplitude (Height)\n"
            "- Frequency (Cycles per second)\n"
            "- Phase (Horizontal shift)"
        )

        eq_general = MathTex(r"y(t) = A \sin(2\pi f t + \phi)", font_size=26).next_to(desc1, DOWN, buff=0.4)
        eq_general.set_color_by_tex("A", YELLOW).set_color_by_tex("f", ORANGE).set_color_by_tex(r"\phi", PURPLE)

        live_eq = always_redraw(lambda: MathTex(
            rf"y(t) = {amp_tracker.get_value():.2f} \sin(2\pi ({freq_tracker.get_value():.2f}) t + {phase_tracker.get_value():.2f})", 
            font_size=24
        ).next_to(eq_general, DOWN, buff=0.3))

        # Dynamic text to show which variable is being tested
        action_text = Text("Base Wave", font_size=20, color=WHITE).next_to(live_eq, DOWN, buff=0.5)

        single_wave = always_redraw(lambda: time_axes.plot(
            lambda x: amp_tracker.get_value() * np.sin(2 * PI * freq_tracker.get_value() * x + phase_tracker.get_value()), 
            color=BLUE
        ))

        self.play(Write(desc1), Write(eq_general))
        self.play(Write(live_eq), Write(action_text), Create(single_wave))
        self.wait(1)
        
        # --- A. INDIVIDUAL: AMPLITUDE ---
        self.play(Transform(action_text, Text("1. Amplitude (A): Controls Height", font_size=20, color=YELLOW).move_to(action_text.get_center())))
        self.play(amp_tracker.animate.set_value(2.5), run_time=1.5)
        self.play(amp_tracker.animate.set_value(0.5), run_time=1.5)
        self.play(amp_tracker.animate.set_value(1.0), run_time=1)
        self.wait(0.5)

        # --- B. INDIVIDUAL: FREQUENCY ---
        self.play(Transform(action_text, Text("2. Frequency (f): Controls Cycles/Speed", font_size=20, color=ORANGE).move_to(action_text.get_center())))
        self.play(freq_tracker.animate.set_value(3.0), run_time=2)
        self.play(freq_tracker.animate.set_value(0.5), run_time=2)
        self.play(freq_tracker.animate.set_value(1.0), run_time=1)
        self.wait(0.5)

        # --- C. INDIVIDUAL: PHASE ---
        self.play(Transform(action_text, Text("3. Phase (Shift): Controls Horiz. Position", font_size=20, color=PURPLE).move_to(action_text.get_center())))
        self.play(phase_tracker.animate.set_value(PI), run_time=1.5)
        self.play(phase_tracker.animate.set_value(-PI), run_time=2)
        self.play(phase_tracker.animate.set_value(0.0), run_time=1.5)
        self.wait(0.5)

        # --- D. MIXED APPROACH ---
        self.play(Transform(action_text, Text("4. Mixed Approach: A, f, and Phase together", font_size=20, color=TEAL).move_to(action_text.get_center())))
        self.play(
            amp_tracker.animate.set_value(1.5),
            freq_tracker.animate.set_value(2.0),
            phase_tracker.animate.set_value(PI/2),
            run_time=3
        )
        self.wait(1)
        
        # Reset back to base for Phase 2
        self.play(
            amp_tracker.animate.set_value(1.0),
            freq_tracker.animate.set_value(1.0),
            phase_tracker.animate.set_value(0.0),
            FadeOut(action_text),
            run_time=1.5
        )

        # ====================
        # 4. PHASE 2: INTERFERENCE (DESTRUCTIVE/CONSTRUCTIVE)
        # ====================
        desc2 = create_desc(
            "2. Wave Interference\n\n"
            "Shifting the phase of a second wave\n"
            "causes Constructive addition or\n"
            "Destructive cancellation."
        )
        
        eq_phase = MathTex(r"y_{sum}(t) = y_1(t) + y_2(t)", color=RED, font_size=26).next_to(desc2, DOWN, buff=0.4)
        
        self.play(FadeOut(desc1, eq_general, live_eq), FadeIn(desc2), Write(eq_phase))

        wave2 = always_redraw(lambda: time_axes.plot(
            lambda x: 1.0 * np.sin(2 * PI * 1.0 * x + phase_tracker.get_value()), color=GREEN
        ))
        sum_wave = always_redraw(lambda: time_axes.plot(
            lambda x: 1.0 * np.sin(2 * PI * 1.0 * x) + 1.0 * np.sin(2 * PI * 1.0 * x + phase_tracker.get_value()), color=RED
        ))

        self.play(Create(wave2))
        self.play(TransformFromCopy(VGroup(single_wave, wave2), sum_wave), run_time=1.5)

        # Destructive (PI shift)
        dest_text = Text("Destructive (Out of Phase)", color=RED, font_size=16).next_to(eq_phase, DOWN)
        self.play(Write(dest_text), phase_tracker.animate.set_value(PI), run_time=2)
        
        # Constructive (0 shift)
        const_text = Text("Constructive (In Phase)", color=RED, font_size=16).next_to(eq_phase, DOWN)
        self.play(ReplacementTransform(dest_text, const_text), phase_tracker.animate.set_value(0), run_time=2)

        self.play(FadeOut(wave2, sum_wave, desc2, eq_phase, const_text))

        # ====================
        # 5. PHASE 3: SUPERPOSITION SWEEP
        # ====================
        desc3 = create_desc(
            "3. Point-by-Point Superposition\n\n"
            "Let's add a 3Hz wave. Watch the\n"
            "vertical line track the exact\n"
            "addition of the amplitudes."
        )
        eq_sum2 = MathTex(r"y_{sum}(t) = \sin(2\pi t) + 0.8\sin(6\pi t)", color=RED, font_size=24).next_to(desc3, DOWN, buff=0.4)
        
        self.play(FadeIn(desc3), Write(eq_sum2))

        wave2_new = time_axes.plot(lambda x: 0.8 * np.sin(2 * PI * 3.0 * x), color=GREEN)
        sum_wave_new = time_axes.plot(lambda x: 1.0 * np.sin(2 * PI * 1.0 * x) + 0.8 * np.sin(2 * PI * 3.0 * x), color=RED)

        self.play(Create(wave2_new))
        self.play(Create(sum_wave_new))

        # Sweeping Tracker Line Animation
        x_val = ValueTracker(0)
        sweep_line = always_redraw(lambda: DashedLine(
            start=time_axes.c2p(x_val.get_value(), 0),
            end=time_axes.c2p(x_val.get_value(), 1.0 * np.sin(2 * PI * 1.0 * x_val.get_value()) + 0.8 * np.sin(2 * PI * 3.0 * x_val.get_value())),
            color=YELLOW
        ))
        sweep_dot = always_redraw(lambda: Dot(
            time_axes.c2p(x_val.get_value(), 1.0 * np.sin(2 * PI * 1.0 * x_val.get_value()) + 0.8 * np.sin(2 * PI * 3.0 * x_val.get_value())),
            color=YELLOW
        ))
        
        self.add(sweep_line, sweep_dot)
        self.play(x_val.animate.set_value(3.5), run_time=4, rate_func=linear)
        self.play(FadeOut(sweep_line, sweep_dot))

        # ====================
        # 6. PHASE 4: FOURIER TRANSFORM
        # ====================
        desc4 = create_desc(
            "4. The Frequency Domain\n\n"
            "A Fourier transform maps the messy\n"
            "time wave into discrete spikes\n"
            "representing pure frequencies."
        )
        eq_transform = MathTex(
            r"\hat{f}(\xi) = \int_{-\infty}^{\infty} f(t) e^{-i 2\pi \xi t} dt", 
            color=YELLOW, font_size=24
        ).next_to(desc4, DOWN, buff=0.4)

        self.play(FadeOut(desc3, eq_sum2), FadeIn(desc4), Write(eq_transform))
        self.play(Create(freq_axes), Write(freq_labels))

        spike1 = VGroup(Line(freq_axes.c2p(1, 0), freq_axes.c2p(1, 1), color=BLUE), Dot(freq_axes.c2p(1, 1), color=BLUE))
        spike3 = VGroup(Line(freq_axes.c2p(3, 0), freq_axes.c2p(3, 0.8), color=GREEN), Dot(freq_axes.c2p(3, 0.8), color=GREEN))

        self.play(GrowFromEdge(spike1, DOWN), GrowFromEdge(spike3, DOWN))
        self.wait(2)
        
        self.play(FadeOut(wave2_new, sum_wave_new, spike1, spike3, single_wave, desc4, eq_transform))

        # ====================
        # 7. PHASE 5: SQUARE WAVE & GIBBS
        # ====================
        desc5 = create_desc(
            "5. Square Wave (Odd Harmonics)\n\n"
            "Summing odd harmonics forms a square.\n"
            "The permanent overshoot at the corners\n"
            "is called the 'Gibbs Phenomenon'."
        )
        eq_sq = MathTex(r"f(t) = \frac{4}{\pi} \sum_{n=1,3...}^{N} \frac{\sin(n \pi t)}{n}", font_size=24).next_to(desc5, DOWN, buff=0.3)
        self.play(FadeIn(desc5), Write(eq_sq))

        colors = [BLUE, GREEN, YELLOW, PURPLE, TEAL, MAROON]
        harmonics_sq = [1, 3, 5, 7, 9, 11]
        
        funcs_sq = []
        all_spikes_sq = VGroup()
        all_harmonic_waves = VGroup()
        
        for i, n in enumerate(harmonics_sq):
            amp = (4 / PI) * (1 / n)
            freq = n / 2.0 
            col = colors[i % len(colors)]

            def create_sq_harmonic(x, a=amp, f=freq): return a * np.sin(2 * PI * f * x)
            funcs_sq.append(create_sq_harmonic)
            
            def get_sq_sum(x, current_funcs=list(funcs_sq)): return sum(f(x) for f in current_funcs)

            harmonic_bg = time_axes.plot(create_sq_harmonic, color=col, stroke_opacity=0.3)
            new_sum_wave = time_axes.plot(get_sq_sum, color=RED)
            spike = VGroup(Line(freq_axes.c2p(n, 0), freq_axes.c2p(n, amp), color=col), Dot(freq_axes.c2p(n, amp), color=col))
            
            all_spikes_sq.add(spike)
            all_harmonic_waves.add(harmonic_bg)

            n_label = Text(f"Adding n={n}", font_size=20, color=col).next_to(eq_sq, DOWN, buff=0.3)

            if i == 0:
                self.play(Create(new_sum_wave), GrowFromEdge(spike, DOWN), Write(n_label), run_time=1)
                current_sum_wave = new_sum_wave
                prev_label = n_label
            else:
                self.play(
                    Transform(current_sum_wave, new_sum_wave), 
                    FadeIn(harmonic_bg),
                    GrowFromEdge(spike, DOWN), 
                    Transform(prev_label, n_label), 
                    run_time=1
                )

        # Highlight Gibbs Phenomenon
        gibbs_arrow = Arrow(start=time_axes.c2p(1, 2.5), end=time_axes.c2p(1, 1.3), color=YELLOW)
        gibbs_text = Text("Gibbs Phenomenon", color=YELLOW, font_size=16).next_to(gibbs_arrow, UP)
        self.play(GrowArrow(gibbs_arrow), Write(gibbs_text))
        self.wait(2)

        self.play(FadeOut(current_sum_wave, all_harmonic_waves, all_spikes_sq, prev_label, desc5, eq_sq, gibbs_arrow, gibbs_text))

        # ====================
        # 8. PHASE 6: SAWTOOTH WAVE
        # ====================
        desc6 = create_desc(
            "6. Sawtooth Wave (All Harmonics)\n\n"
            "Including ALL integer harmonics and\n"
            "alternating their signs (-/+) creates\n"
            "a Sawtooth wave!"
        )
        eq_saw = MathTex(r"f(t) = \frac{2}{\pi} \sum_{n=1}^{N} (-1)^{n+1} \frac{\sin(n \pi t)}{n}", font_size=24).next_to(desc6, DOWN, buff=0.3)
        self.play(FadeIn(desc6), Write(eq_saw))

        harmonics_saw = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
        funcs_saw = []
        
        for i, n in enumerate(harmonics_saw):
            # Alternating sign mathematically applied
            amp = (2 / PI) * (1 / n) * ((-1)**(n+1))
            freq = n / 2.0 
            col = colors[i % len(colors)]

            def create_saw_harmonic(x, a=amp, f=freq): return a * np.sin(2 * PI * f * x)
            funcs_saw.append(create_saw_harmonic)
            
            def get_saw_sum(x, current_funcs=list(funcs_saw)): return sum(f(x) for f in current_funcs)

            new_sum_wave = time_axes.plot(get_saw_sum, color=ORANGE)
            spike = VGroup(Line(freq_axes.c2p(n, 0), freq_axes.c2p(n, abs(amp)), color=col), Dot(freq_axes.c2p(n, abs(amp)), color=col))

            n_label = Text(f"Adding n={n}", font_size=20, color=col).next_to(eq_saw, DOWN, buff=0.3)

            if i == 0:
                self.play(Create(new_sum_wave), GrowFromEdge(spike, DOWN), Write(n_label), run_time=1)
                current_sum_wave = new_sum_wave
                prev_label = n_label
            else:
                self.play(Transform(current_sum_wave, new_sum_wave), GrowFromEdge(spike, DOWN), Transform(prev_label, n_label), run_time=0.8)

        self.wait(3)

%manim -ql -v warning FourierConcept

Manim Community v0.19.0